# [9665] Latent Dirichlet Allocation 1

Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Medium_articles.csv

In [1]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 02/23/25 19:52:51


### Import libraries

In [2]:
! pip install pyldavis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 15.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import nltk
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
import pyLDAvis
import pyLDAvis.gensim

import warnings
# Suppress the DeprecationWarning
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [4]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

### Load data

In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Medium_articles.csv')
df.shape

(337, 6)

### Review data

In [6]:
pd.set_option('max_colwidth', None)

In [7]:
df.head(2)

author claps  reading_time  \
0   Justin Lee  8.3K            11   
1  Conor Dewey  1.4K             7   

                                                                                                                                            link  \
0                          https://medium.com/swlh/chatbots-were-the-next-big-thing-what-happened-5fc49dd6fa61?source=---------0----------------   
1  https://towardsdatascience.com/python-for-data-science-8-concepts-you-may-have-forgotten-i-did-825966908393?source=---------1----------------   

                                                                     title  \
0  Chatbots were the next big thing: what happened? – The Startup – Medium   
1               Python for Data Science: 8 Concepts You May Have Forgotten   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [8]:
# Create new column to perform LDA on
df['combined_text'] = df['title'] + ' ' + df['text']
df[['title', 'text', 'combined_text']].head(2)

title  \
0  Chatbots were the next big thing: what happened? – The Startup – Medium   
1               Python for Data Science: 8 Concepts You May Have Forgotten   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

### Preprocess data

In [9]:
# Save sample article index for later examination
SAMPLE_ARTICLE_1_IDX = 301

In [10]:
# Create preprocessing functions
def lemmatizer(text):
    return WordNetLemmatizer().lemmatize(text, pos='v')

def preprocess(text):
    result = []
    for token in simple_preprocess(text):
        if token not in STOPWORDS and len(token) > 3:
            result.append(lemmatizer(token))
    return result

In [11]:
# Show results of stemmer for sample document
example_doc = df.iloc[SAMPLE_ARTICLE_1_IDX]['combined_text']
print('original document:\n\t{}'.format(example_doc))
print('tokenized and lemmatized document:\n\t{}'.format(preprocess(example_doc)))

original document:
	Everything You Need to Know About Artificial Neural Networks The year 2015 was a monumental year in the field of artificial intelligence. Not only are computers learning more and learning faster, but we’re learning more about how to improve their systems. Everything is starting to align, and because of it we’re seeing strides we’ve never thought possible until now. We have programs that can tell stories about pictures. We have cars that are driving themselves. We even have programs that create art. If you want to read more about advancements in 2015, read this article. Here at Josh.ai, with AI technology becoming the core of just about everything we do, we think it’s important to understand some of the common terminology and to get a rough idea of how it all works.
A lot of the advances in artificial intelligence are new statistical models, but the overwhelming majority of the advances are in a technology called artificial neural networks (ANN). If you’ve read anyth

In [12]:
%%time

# Preprocess all documents
processed_docs = df['combined_text'].map(preprocess)

CPU times: user 3.9 s, sys: 48.9 ms, total: 3.95 s
Wall time: 7.92 s


In [13]:
# Review first few processed documents
processed_docs.head()

,combined_text
0,"[chatbots, thing, happen, startup, medium, headline, blare, chatbots, thing, hop, high, bright, eye, bushy, tail, industry, ripe, innovation, time, start, socialize, machine, wouldn, road, sign, point, insane, success, mobile, world, congress, chatbots, main, headliners, conference, organizers, cite, overwhelm, acceptance, event, inevitable, shift, focus, brand, corporates, chatbots, fact, significant, question, chatbots, monopolize, field, chatbots, place, year, answer, question, ecosystem, platform, dominate, chatbots, weren, technological, development, talk, grandiose, term, slump, spectacularly, hype, cycle, unfold, familiar, fashion, expectations, build, build, kind, fizzle, predict, paradim, shift, materialize, apps, tellingly, alive, look, breathless, optimism, turn, slightly, baffle, chatbot, revolution, promise, digit, ethan, bloch, sum, general, ...]"
1,"[python, data, science, concepts, forget, look, question, concept, syntax, program, constantly, unnatural, look, things, stackoverflow, resources, slow, good, raise, question, complete, understand, language, live, world, seemingly, infinite, accessible, free, resources, loom, search, away, time, bless, curse, manage, effectively, reliance, resources, build, poor, habit, long, term, personally, pull, code, similar, discussion, thread, time, take, time, learn, solidify, concept, reproduce, code, time, approach, lazy, path, resistance, short, term, ultimately, hurt, growth, productivity, ability, recall, syntax, cough, interview, line, recently, work, online, data, science, course, title, python, data, science, machine, learn, udemy, sound, like, youtube, early, lecture, series, remind, concepts, syntax, consistently, overlook, ...]"
2,"[automate, feature, engineer, python, data, science, machine, learn, increasingly, move, hand, design, model, automatically, optimize, pipelines, tool, tpot, auto, sklearn, libraries, methods, random, search, simplify, model, selection, tune, part, machine, learn, find, best, model, dataset, little, manual, intervention, feature, engineer, arguably, valuable, aspect, machine, learn, pipeline, remain, entirely, human, labor, feature, engineer, know, feature, creation, process, construct, feature, exist, data, train, machine, learn, model, step, important, actual, model, machine, learn, algorithm, learn, data, create, feature, relevant, task, absolutely, crucial, excellent, paper, useful, things, know, machine, learn, typically, feature, engineer, draw, manual, process, rely, domain, knowledge, intuition, data, manipulation, process, extremely, ...]"
3,"[machine, learn, zero, hero, freecodecamp, understand, machine, learn, question, mark, blog, post, gradually, increase, glue, inspirational, videos, friendly, text, relax, videos, time, inspire, continue, section, fair, article, earn, round, knowledge, passion, world, cool, move, paddle, pong, light, combos, street, fighter, revolve, programmer, functional, guess, behave, programmers, aren, gift, program, google, epic, game, fail, glitches, physics, experience, human, players, regardless, talent, teach, play, video, game, understand, language, identify, people, things, iceberg, skill, come, concept, recently, process, power, exist, outside, theory, talk, machine, learn, need, come, advance, algorithms, anymore, teach, come, advance, algorithm, like, work, algorithm, write, sort, breed, breed, analogy, watch, ...]"
4,"[reinforcement, learn, scratch, insight, data, want, learn, apply, artificial, intelligence, lead, practitioners, silicon, valley, york, toronto, learn, insight, artificial, intelligence, fellows, program, company, work, like, involve, insight, fellows, program, feel, free, touch, recently, give, talk, reilly, conference, beijing, interest, lessons, learn, world, lucky, attend, tutorial, deep, reinforcement, learn, deep, scratch, unity, technologies, think, session, arthur, juliani, extremely, informative, want, share, takeaways, conversations, company, see, r

### Generate Gensim Dictionary object

In [14]:
%%time

# Map each word in ‘processed_docs’ to its unique integer id (index)
dictionary = Dictionary(processed_docs)

CPU times: user 445 ms, sys: 1.87 ms, total: 447 ms
Wall time: 1.18 s


In [15]:
print("{} words in Dictionary object".format(len(dictionary)))

14614 words in Dictionary object


In [16]:
# Display first 10 elements in dictionary
for i, (k, v) in enumerate(dictionary.items()):
    print(k, v)
    if i >= 10:
        break

0 able
1 abruptly
2 acceptance
3 access
4 accident
5 accord
6 achieve
7 adage
8 adopters
9 advantage
10 agenda


In [17]:
# Remove very rare and very common words
#  Filter out tokens that appear in
#   < 15 documents (absolute number) or
#   > 50% documents (fraction of total corpus size, not absolute number)
#  After the above two steps, keep only the first 10000 most frequent tokens
#   Next, shrink resulting gaps in word ids
dictionary.filter_extremes(no_below=15, no_above=0.5, keep_n=10000)
print("{} words remaining in Dictionary object".format(len(dictionary)))

1886 words remaining in Dictionary object


In [18]:
# Display first 10 elements in updated dictionary
for i, (k, v) in enumerate(dictionary.items()):
    print(k, v)
    if i >= 10:
        break

0 access
1 accord
2 achieve
3 advantage
4 aim
5 allow
6 alternative
7 analyze
8 answer
9 anticipate
10 apply


In [19]:
%%time

# Convert documents into the bag-of-words format: list of (token_id, token_count) 2-tuples
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

CPU times: user 254 ms, sys: 2.06 ms, total: 256 ms
Wall time: 1.01 s


In [20]:
# Check processed BoW version of sample document
bow_corpus[SAMPLE_ARTICLE_1_IDX]

[(0, 1),
 (2, 1),
 (5, 1),
 (8, 4),
 (10, 2),
 (24, 2),
 (32, 2),
 (56, 1),
 (59, 1),
 (64, 3),
 (103, 2),
 (113, 1),
 (116, 1),
 (118, 1),
 (120, 2),
 (122, 1),
 (124, 4),
 (135, 2),
 (136, 1),
 (143, 2),
 (147, 2),
 (149, 1),
 (155, 2),
 (158, 1),
 (164, 3),
 (166, 3),
 (171, 5),
 (175, 1),
 (177, 1),
 (184, 1),
 (191, 5),
 (195, 4),
 (200, 1),
 (208, 3),
 (209, 1),
 (211, 16),
 (212, 1),
 (222, 1),
 (228, 1),
 (229, 1),
 (230, 1),
 (232, 1),
 (238, 1),
 (239, 2),
 (242, 1),
 (243, 2),
 (246, 1),
 (251, 2),
 (253, 1),
 (255, 1),
 (259, 1),
 (261, 6),
 (262, 1),
 (266, 2),
 (267, 1),
 (270, 2),
 (271, 1),
 (274, 2),
 (285, 2),
 (292, 4),
 (293, 1),
 (296, 2),
 (314, 1),
 (321, 5),
 (323, 1),
 (328, 2),
 (339, 1),
 (342, 1),
 (346, 1),
 (347, 1),
 (349, 1),
 (355, 1),
 (358, 1),
 (360, 1),
 (361, 3),
 (363, 1),
 (365, 3),
 (366, 1),
 (368, 2),
 (371, 1),
 (382, 2),
 (384, 1),
 (385, 3),
 (386, 2),
 (387, 1),
 (389, 1),
 (394, 1),
 (399, 5),
 (402, 1),
 (403, 2),
 (405, 2),
 (407, 1),
 

In [21]:
# Review sample document in B0W format
bow_example_doc = bow_corpus[SAMPLE_ARTICLE_1_IDX]

print('word_index\tword\t\tword_occurences')
for i in range(len(bow_example_doc)):
    print(f'{bow_example_doc[i][0]}\t\t{dictionary[bow_example_doc[i][0]]}\t\t{bow_example_doc[i][1]}')

word_index	word		word_occurences
0		access		1
2		achieve		1
5		allow		1
8		answer		4
10		apply		2
24		basic		2
32		brain		2
56		company		1
59		complete		1
64		computers		3
103		early		2
113		email		1
116		entirely		1
118		error		1
120		essentially		2
122		eventually		1
124		examples		4
135		facebook		2
136		fact		1
143		faster		2
147		field		2
149		finish		1
155		forward		2
158		general		1
164		happen		3
166		have		3
171		hide		5
175		home		1
177		https		1
184		imagine		1
191		information		5
195		intelligence		4
200		internet		1
208		language		3
209		larger		1
211		layer		16
212		lead		1
222		linear		1
228		long		1
229		longer		1
230		love		1
232		majority		1
238		memory		1
239		mention		2
242		middle		1
243		mind		2
246		modern		1
251		natural		2
253		necessarily		1
255		obviously		1
259		originally		1
261		output		6
262		overwhelm		1
266		picture		2
267		place		1
270		play		2
271		plenty		1
274		power		2
285		provide		2
292		question		4
293		quickly		1
296		reason		2
314		sake		1
321

### Train LDA model

In [22]:
%%time

lda_model = LdaModel(corpus=bow_corpus, id2word=dictionary, num_topics=10,
                     chunksize=10, passes=10, alpha='auto', eta='auto',
                     gamma_threshold=0.001, per_word_topics=True, random_state=42)

CPU times: user 6.76 s, sys: 73.8 ms, total: 6.84 s
Wall time: 16.7 s


### Evaluate LDA model

In [23]:
%%time

# Compute Coherence Score: c_v
coherence_model_lda = CoherenceModel(model=lda_model, texts=processed_docs,
                                     dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('Coherence Score: ', coherence_lda)

Coherence Score:  0.47228836260580176
CPU times: user 9.91 s, sys: 142 ms, total: 10.1 s
Wall time: 10.3 s


In [24]:
%%time

# Compute Coherence Score: u_mass
coherence_model_lda = CoherenceModel(model=lda_model, texts=processed_docs,
                                     dictionary=dictionary, coherence='u_mass')
coherence_lda = coherence_model_lda.get_coherence()
print('Coherence Score: ', coherence_lda)

Coherence Score:  -1.0977017868328844
CPU times: user 155 ms, sys: 126 µs, total: 156 ms
Wall time: 156 ms


In [25]:
# Get the gamma matrix (document-topic matrix)
gamma, _ = lda_model.inference(bow_corpus)

# Display the gamma values
for i, gamma_values in enumerate(gamma):
    print(f"Document {i+1}: {gamma_values}")

Document 1: [1.2199944e+01 9.2726761e+01 1.3182793e+02 1.8802855e+02 1.8803070e+02
 9.1095352e-01 1.7024544e+01 7.7462912e+00 2.1276281e+00 1.2470807e-01]
Document 2: [91.58197   24.551872  61.479774   0.3270069 27.398998  83.69977
 16.345642  41.429535  24.881004   0.1247493]
Document 3: [190.64886     44.028126    25.464064     0.28611803  84.50818
  62.825123   137.60521     53.927326    24.973427     1.463624  ]
Document 4: [  2.1672626   2.9529145 117.42148     5.7219043  67.516396   61.043873
   1.7238411  33.32842    17.658821    7.29273  ]
Document 5: [180.70508   29.322933  39.668633   4.322441  68.96046    8.427811
  37.657352  16.07118    7.818147 295.72717 ]
Document 6: [1.57757294e+02 5.45648232e+01 9.39603806e+00 3.10409129e-01
 2.30033836e+01 7.36554480e+00 5.79392883e+02 3.35479469e+01
 1.11156158e+02 1.24625415e-01]
Document 7: [ 8.254444 34.157276 43.85752  29.44574  37.643715 12.031549 33.688515
 47.253075 42.267426 21.225203]
Document 8: [9.03857422e+00 5.91074610e+

In [26]:
# Display topics identified by LDA model
topics = lda_model.print_topics()
for topic in topics:
    print(topic)

(0, '0.040*"function" + 0.024*"neuron" + 0.021*"value" + 0.018*"gradient" + 0.017*"loss" + 0.011*"output" + 0.011*"vector" + 0.010*"sigmoid" + 0.010*"table" + 0.009*"zero"')
(1, '0.014*"architecture" + 0.013*"performance" + 0.010*"test" + 0.010*"provide" + 0.009*"compute" + 0.009*"run" + 0.009*"perform" + 0.009*"improve" + 0.008*"power" + 0.008*"similar"')
(2, '0.012*"word" + 0.011*"memory" + 0.009*"say" + 0.009*"google" + 0.008*"generate" + 0.008*"brain" + 0.007*"text" + 0.007*"language" + 0.007*"article" + 0.007*"turn"')
(3, '0.026*"say" + 0.025*"bots" + 0.024*"message" + 0.022*"sound" + 0.019*"chatbots" + 0.017*"user" + 0.017*"chatbot" + 0.017*"felt" + 0.014*"person" + 0.013*"users"')
(4, '0.023*"intelligence" + 0.021*"artificial" + 0.017*"systems" + 0.017*"technology" + 0.012*"company" + 0.011*"years" + 0.010*"field" + 0.010*"research" + 0.010*"software" + 0.010*"future"')
(5, '0.034*"course" + 0.030*"python" + 0.028*"https" + 0.020*"blog" + 0.018*"program" + 0.018*"keras" + 0.017*

In [27]:
# Alternatively, use show_topics, which gives more options
topics = lda_model.show_topics(num_topics=5, num_words=5, formatted=False)
for topic in topics:
    print(topic)

(9, [('action', 0.09506562), ('game', 0.05293879), ('reinforcement', 0.046531457), ('state', 0.045139953), ('policy', 0.042585287)])
(3, [('say', 0.025678363), ('bots', 0.025151536), ('message', 0.024490038), ('sound', 0.022419864), ('chatbots', 0.019159501)])
(1, [('architecture', 0.014061382), ('performance', 0.012969055), ('test', 0.010217222), ('provide', 0.009984964), ('compute', 0.009482685)])
(0, [('function', 0.03959323), ('neuron', 0.023712989), ('value', 0.020831168), ('gradient', 0.018119125), ('loss', 0.017020198)])
(2, [('word', 0.011774831), ('memory', 0.01081743), ('say', 0.009354277), ('google', 0.008912241), ('generate', 0.008229118)])


In [28]:
# Create Topic Distance Visualization
pyLDAvis.enable_notebook()
pyLDAvis.gensim.prepare(lda_model, bow_corpus, dictionary)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2      0.040412 -0.120814       1        1  23.658050
4      0.189940  0.097741       2        1  13.503945
0     -0.154160  0.146854       3        1  12.449466
1     -0.018036  0.264722       4        1  11.358462
6     -0.171687  0.074517       5        1  10.712311
7      0.111053 -0.153142       6        1   8.526427
5      0.006936 -0.244534       7        1   7.548246
8     -0.281569 -0.107960       8        1   5.176954
3      0.251457  0.060833       9        1   4.232831
9      0.025655 -0.018216      10        1   2.833309, topic_info=           Term         Freq        Total Category  logprob  loglift
846       image  1617.000000  1617.000000  Default  30.0000  30.0000
211       layer  1123.000000  1123.000000  Default  29.0000  29.0000
776      weight   843.000000   843.000000  Default  28.0000  28.0000
450    function  1067.000000  1067.000000  Default  27.0000  27.0000
546      action   459.000000   459.000000  Default  26.0000  26.0000
..          ...          ...          ...      ...      ...      ...
285     provide    59.086208   397.128512  Topic10  -4.4021   1.6585
684  algorithms    51.916313   348.510857  Topic10  -4.5315   1.6597
126      expect    40.346722   200.539142  Topic10  -4.7836   1.9602
563      choose    42.042137   246.367458  Topic10  -4.7425   1.7956
502     science    40.438775   423.524319  Topic10  -4.7813   1.2149

[578 rows x 6 columns], token_table=      Topic      Freq      Term
term                           
1296      4  0.188907  accuracy
1296      5  0.809013  accuracy
2         1  0.211133   achieve
2         2  0.015449   achieve
2         3  0.041197   achieve
...     ...       ...       ...
404       5  0.185151     years
404       6  0.087571     years
543       7  0.978753   youtube
778       1  0.082251      zero
778       3  0.913902      zero

[1097 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 5, 1, 2, 7, 8, 6, 9, 4, 10])